In [1]:
import ast
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

datasets = Path("/nas/cee-water/cjgleason/data")
era5_dir = datasets / "ERA5-Land/sub_basin_timeseries"
swot_lake_dir = datasets / 'hydrocron' / 'lake'

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/reservoirs")
preprocess_dir = save_dir / "preprocess"
metadata_dir = save_dir / "metadata"

matchups = gpd.read_parquet(metadata_dir / "All_MERIT_matchups.parquet").set_index('comid')
matchups.index = matchups.index.astype(str)

In [2]:
from data import BasinDataLake

root_dir = save_dir / 'datalakes' / 'training'
store = BasinDataLake(root_dir)

processed_basins = store.get_processing_status(source='gauge')
to_process = matchups[~matchups.index.isin(processed_basins['subbasin'])]
to_process

,outlet,outlet_id,total_area,unitarea,reservoir,custom,reach_id,sword_area,sword_distance,lake_reach_ids,...,longitude,min_date,max_date,min_discharge,max_discharge,mean_discharge,count_discharge,provider,hybas_area_diff,geometry
comid,,,,,,,,,,,,,,,,,,,,,
21000001,POINT (5.889166666666659 47.94916666666667),EAUF-V7200010,398.5,152.9,False,False,NaN,NaN,NaN,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,-0.034201,"MULTIPOLYGON (((5.93875 47.99125, 5.93708 47.9..."
21000012,POINT (5.889166666666659 47.94916666666667),EAUF-V7200010,324.8,194.6,False,False,NaN,NaN,NaN,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,0.053341,"MULTIPOLYGON (((5.79042 48.01125, 5.79042 48.0..."
21000019,POINT (5.684999999999993 47.53416666666667),EAUF-V7200010,5173.0,242.5,False,False,2.160280e+10,4530.071328,0.0,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,-0.007977,"MULTIPOLYGON (((5.76042 47.56125, 5.76042 47.5..."
21000021,POINT (5.76916666666666 47.58083333333334),EAUF-V7200010,4758.7,8.4,False,False,2.160280e+10,4278.612301,0.0,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,0.003021,"MULTIPOLYGON (((5.80625 47.60125, 5.80708 47.6..."
21000022,POINT (5.80416666666666 47.57),EAUF-V7200010,4506.6,68.9,False,False,2.160280e+10,4278.612301,0.0,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,0.009101,"MULTIPOLYGON (((5.85875 47.57458, 5.85875 47.5..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
USGS-410401112134801,POINT (-112.1875 41.0625),USGS-410401112134801,9900.3,296.5,False,True,NaN,NaN,NaN,[],...,-112.230256,2003-10-01,2025-09-08,0.006230,45.873291,11.263905,7165.0,usgs,-0.002093,"MULTIPOLYGON (((-112.21458 40.96042, -112.2154..."
USGS-50035000,POINT (-66.4592 18.3217),USGS-50037000,345.7,345.7,False,True,NaN,NaN,NaN,[],...,-66.459568,1950-01-01,2025-09-08,0.240693,1857.585136,7.406389,25691.0,usgs,-0.091982,"MULTIPOLYGON (((-66.49208 18.29625, -66.49208 ..."
USGS-50037000,POINT (-66.5 18.3983),USGS-50037000,429.1,83.4,False,True,NaN,NaN,NaN,[],...,-66.496560,2019-06-13,2025-09-08,1.265763,911.802460,11.017469,2247.0,usgs,0.167805,"MULTIPOLYGON (((-66.50125 18.33125, -66.50125 ..."


In [3]:
import global_gauges as gg

facade = gg.GaugeDataFacade()
sites = facade.get_stations_n_days(90)

/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/global_gauges/facade.py:192: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(provider_info)


In [4]:
processed_basins = store.get_processing_status(source='gauge')
to_process = matchups[~matchups.index.isin(processed_basins['subbasin'])]

max_batch_size = 256
for basin_id, basin_group in tqdm(to_process.groupby('outlet_id')):
    subbasin_data = {}
    for comid, row in basin_group.iterrows():
        if not row['custom']:
            gauge_df = None
        else:
            # There is not an explicit flag for custom that is not a gauge, so we just try
            # and then catch the error if this comid doesn't exist in our gauge dataset.
            try:
                gauge_df = facade.get_daily_values(comid).droplevel(axis=0, level=0)
            except ValueError:
                gauge_df = None
                
        subbasin_data[comid] = gauge_df

        if len(subbasin_data)==max_batch_size:
            store.write_dynamic(basin_id, 'gauge', subbasin_data, mode='append')
            subbasin_data = {}

    # Write any remaining data.
    if len(subbasin_data)>0:
        store.write_dynamic(basin_id, 'gauge', subbasin_data, mode='append')

  4%|▎         | 23/643 [01:49<2:01:41, 11.78s/it][2025-09-20T16:41:21Z WARN  deltalake_core::kernel::transaction] Attempting to write a transaction 405 but the underlying table has been updated to 405
    DefaultLogStore(/nas/cee-water/cjgleason/ted/swot-ml/data/reservoirs/datalakes/training/processing_metadata/)
 22%|██▏       | 144/643 [30:39<45:44,  5.50s/it][2025-09-20T17:10:10Z WARN  deltalake_core::kernel::transaction] Attempting to write a transaction 1356 but the underlying table has been updated to 1356
    DefaultLogStore(/nas/cee-water/cjgleason/ted/swot-ml/data/reservoirs/datalakes/training/processing_metadata/)
 25%|██▍       | 158/643 [32:13<1:17:47,  9.62s/it][2025-09-20T17:11:43Z WARN  deltalake_core::kernel::transaction] Attempting to write a transaction 1394 but the underlying table has been updated to 1394
    DefaultLogStore(/nas/cee-water/cjgleason/ted/swot-ml/data/reservoirs/datalakes/training/processing_metadata/)
100%|██████████| 643/643 [3:33:22<00:00, 19.91s/

In [ ]:
row['custom']

In [ ]:
processed_basins = store.get_processing_status(source='swot-river')
processed_basins

In [ ]:
processed_basins[processed_basins['has_data']]['basin'].value_counts()

In [ ]:
store.compact()

In [ ]:
matchups['lake_pld_ids'].apply(len).gt(0).sum()